# Query Tarantino — Sprint 1 Benchmark Analysis

Análisis reproducible de los resultados de los benchmarks de Sprint 1.

Los resultados se leen exclusivamente desde:

`benchmarks/results/`

## Experimentos

- Datalake: `time`, `book`, `batch`
- Índices: `json`, `mongo`, `folders`
- Descarga: `none`

## Lenguajes

- Python
- Java
- C++

## Tamaños del corpus

- 100 libros
- 250 libros
- 500 libros
- 1000 libros

## Runs

- `run=0`: warm-up, no forma parte de los resultados oficiales.
- `run=1,2,3`: ejecuciones oficiales utilizadas para el análisis.

Los CSV originales no se modifican durante este análisis.

## 1. Imports y configuración

In [ ]:
from pathlib import Path

import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Localización del proyecto y de los resultados.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "benchmarks":
    PROJECT_ROOT = CURRENT_DIR.parent
    RESULTS_DIR = CURRENT_DIR / "results"
else:
    PROJECT_ROOT = CURRENT_DIR
    RESULTS_DIR = PROJECT_ROOT / "benchmarks" / "results"

print(f"Project root: {PROJECT_ROOT}")
print(f"Results directory: {RESULTS_DIR}")

if not RESULTS_DIR.exists():
    raise FileNotFoundError(
        f"No existe el directorio de resultados: {RESULTS_DIR}"
    )

## 2. Contrato de los resultados

El formato de cada CSV debe ser exactamente:

`language,experiment,structure,n_books,metric,value,unit,run`

Los resultados oficiales contienen únicamente los runs `1`, `2` y `3`.

El warm-up `run=0` se ejecuta durante el benchmark, pero no se incluye en los CSV oficiales analizados aquí.

In [ ]:
EXPECTED_COLUMNS = [
    "language",
    "experiment",
    "structure",
    "n_books",
    "metric",
    "value",
    "unit",
    "run",
]

LANGUAGES = {
    "python",
    "java",
    "cpp",
}

N_BOOKS = {
    100,
    250,
    500,
    1000,
}

OFFICIAL_RUNS = {
    1,
    2,
    3,
}

STRUCTURES = {
    "datalake": {
        "time",
        "book",
        "batch",
    },
    "index": {
        "json",
        "mongo",
        "folders",
    },
    "download": {
        "none",
    },
}

METRICS = {
    "datalake": {
        "write_throughput": "books_per_s",
        "lookup_scan_mean": "ms",
        "lookup_metadata_mean": "ms",
        "detect_new": "ms",
        "recovery_correct": "ok",
        "recovery_time": "ms",
        "disk_bytes": "bytes",
        "file_count": "files",
        "dir_count": "dirs",
        "peak_rss": "bytes",
    },
    "index": {
        "build_time": "ms",
        "query_mean": "ms",
        "query_p95": "ms",
        "update_50_time": "ms",
        "disk_bytes": "bytes",
        "peak_rss": "bytes",
    },
    "download": {
        "http_throughput": "books_per_s",
        "peak_rss": "bytes",
    },
}

EXPECTED_FILES = [
    f"{language}_{experiment}.csv"
    for language in sorted(LANGUAGES)
    for experiment in sorted(STRUCTURES)
]

EXPECTED_FILES

## 3. Carga de los resultados

Se cargan los nueve CSV oficiales:

- `python_datalake.csv`
- `python_index.csv`
- `python_download.csv`
- `java_datalake.csv`
- `java_index.csv`
- `java_download.csv`
- `cpp_datalake.csv`
- `cpp_index.csv`
- `cpp_download.csv`

In [ ]:
results = {}

for language in sorted(LANGUAGES):
    for experiment in sorted(STRUCTURES):
        filename = f"{language}_{experiment}.csv"
        path = RESULTS_DIR / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Falta el resultado oficial: {path}"
            )

        df = pd.read_csv(path)

        results[(language, experiment)] = df

        print(
            f"{filename}: {len(df)} filas"
        )

## 4. Validación de los resultados

Antes de realizar cualquier análisis se comprueba que los CSV contienen únicamente configuraciones y métricas definidas por el SPEC.

Estas comprobaciones son adicionales a `benchmarks/scripts/validate_csv.py`.

In [ ]:
def validate_dataframe(
    df: pd.DataFrame,
    language: str,
    experiment: str,
) -> None:

    filename = f"{language}_{experiment}.csv"

    # Columnas
    if list(df.columns) != EXPECTED_COLUMNS:
        raise ValueError(
            f"{filename}: columnas incorrectas.\n"
            f"Esperadas: {EXPECTED_COLUMNS}\n"
            f"Encontradas: {list(df.columns)}"
        )

    # Language
    if set(df["language"]) != {language}:
        raise ValueError(
            f"{filename}: language incorrecto."
        )

    # Experiment
    if set(df["experiment"]) != {experiment}:
        raise ValueError(
            f"{filename}: experiment incorrecto."
        )

    # n_books
    if not set(df["n_books"]).issubset(N_BOOKS):
        raise ValueError(
            f"{filename}: contiene tamaños no permitidos."
        )

    # Runs
    if not set(df["run"]).issubset(OFFICIAL_RUNS):
        raise ValueError(
            f"{filename}: contiene runs no oficiales: "
            f"{sorted(set(df['run']) - OFFICIAL_RUNS)}"
        )

    # Structures
    expected_structures = STRUCTURES[experiment]

    if not set(df["structure"]).issubset(expected_structures):
        raise ValueError(
            f"{filename}: contiene estructuras no permitidas."
        )

    # Metrics
    expected_metrics = METRICS[experiment]

    if not set(df["metric"]).issubset(expected_metrics):
        raise ValueError(
            f"{filename}: contiene métricas no permitidas."
        )

    # Units
    for metric, expected_unit in expected_metrics.items():
        metric_rows = df[df["metric"] == metric]

        if metric_rows.empty:
            continue

        actual_units = set(metric_rows["unit"])

        if actual_units != {expected_unit}:
            raise ValueError(
                f"{filename}: unidad incorrecta para "
                f"{metric}: {actual_units}"
            )

    # Numeric values
    values = pd.to_numeric(
        df["value"],
        errors="coerce",
    )

    if values.isna().any():
        raise ValueError(
            f"{filename}: existen valores no numéricos."
        )

    if not np.isfinite(values).all():
        raise ValueError(
            f"{filename}: existen valores no finitos."
        )

    # Duplicates
    key_columns = [
        "language",
        "experiment",
        "structure",
        "n_books",
        "metric",
        "run",
    ]

    if df.duplicated(subset=key_columns).any():
        raise ValueError(
            f"{filename}: existen resultados duplicados."
        )

In [ ]:
for (language, experiment), df in results.items():
    validate_dataframe(
        df,
        language,
        experiment,
    )

print("OK: todos los CSV cumplen el contrato básico del SPEC.")

## 5. Comprobación de cobertura experimental

Cada combinación esperada debe disponer de los tres runs oficiales y de todas las métricas definidas para su experimento.

In [ ]:
def validate_coverage(
    df: pd.DataFrame,
    language: str,
    experiment: str,
) -> None:

    filename = f"{language}_{experiment}.csv"

    expected_structures = STRUCTURES[experiment]
    expected_metrics = METRICS[experiment]

    expected_keys = {
        (
            structure,
            n_books,
            metric,
            run,
        )
        for structure in expected_structures
        for n_books in N_BOOKS
        for metric in expected_metrics
        for run in OFFICIAL_RUNS
    }

    actual_keys = {
        (
            row.structure,
            row.n_books,
            row.metric,
            row.run,
        )
        for row in df.itertuples()
    }

    missing = expected_keys - actual_keys
    extra = actual_keys - expected_keys

    if missing:
        raise ValueError(
            f"{filename}: faltan {len(missing)} resultados."
        )

    if extra:
        raise ValueError(
            f"{filename}: existen {len(extra)} resultados inesperados."
        )

In [ ]:
for (language, experiment), df in results.items():
    validate_coverage(
        df,
        language,
        experiment,
    )

print("OK: cobertura experimental completa.")

## 6. Estadísticas oficiales

Para cada configuración experimental se calcula la mediana de los tres runs oficiales:

`run=1,2,3`

El `run=0` no participa en estas estadísticas.

In [ ]:
def median_results(df: pd.DataFrame) -> pd.DataFrame:
    official = df[
        df["run"].isin(OFFICIAL_RUNS)
    ].copy()

    group_columns = [
        "language",
        "experiment",
        "structure",
        "n_books",
        "metric",
        "unit",
    ]

    return (
        official
        .groupby(group_columns, as_index=False)["value"]
        .median()
        .rename(
            columns={
                "value": "median_value"
            }
        )
    )

In [ ]:
median_results_by_experiment = {}

for key, df in results.items():
    median_results_by_experiment[key] = median_results(df)

all_results = pd.concat(
    median_results_by_experiment.values(),
    ignore_index=True,
)

all_results.head()

# 7. Datalake

Se analizan las estructuras:

- `time`
- `book`
- `batch`

y las métricas definidas por el SPEC.

In [ ]:
datalake_results = all_results[
    all_results["experiment"] == "datalake"
].copy()

datalake_results

In [ ]:
write_df = datalake_results[
    datalake_results["metric"] == "write_throughput"
].copy()

write_df

In [ ]:
def plot_scalability(
    df: pd.DataFrame,
    title: str,
    ylabel: str,
) -> None:

    if df.empty:
        print("No hay datos para representar.")
        return

    for language in sorted(df["language"].unique()):

        language_df = df[
            df["language"] == language
        ]

        plt.figure()

        for structure in sorted(
            language_df["structure"].unique()
        ):

            structure_df = (
                language_df[
                    language_df["structure"] == structure
                ]
                .sort_values("n_books")
            )

            plt.plot(
                structure_df["n_books"],
                structure_df["median_value"],
                marker="o",
                label=structure,
            )

        plt.xlabel("Number of books")
        plt.ylabel(ylabel)
        plt.title(f"{title} — {language}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [ ]:
plot_scalability(
    write_df,
    "Datalake write throughput",
    "Books per second",
)

## 7.1 Localización de libros

Se comparan:

- `lookup_scan_mean`
- `lookup_metadata_mean`

In [ ]:
lookup_df = datalake_results[
    datalake_results["metric"].isin([
        "lookup_scan_mean",
        "lookup_metadata_mean",
    ])
].copy()

lookup_df

In [ ]:
for metric in [
    "lookup_scan_mean",
    "lookup_metadata_mean",
]:
    plot_scalability(
        lookup_df[
            lookup_df["metric"] == metric
        ],
        metric,
        "Milliseconds",
    )

## 7.2 Detección de nuevos libros

Métrica:

- `detect_new`

In [ ]:
detect_new_df = datalake_results[
    datalake_results["metric"] == "detect_new"
].copy()

plot_scalability(
    detect_new_df,
    "Datalake new-book detection time",
    "Milliseconds",
)

## 7.3 Recuperación

Métricas:

- `recovery_correct`
- `recovery_time`

`recovery_correct` debe representar una recuperación correcta mediante `0.0` o `1.0`.

In [ ]:
recovery_correct_df = datalake_results[
    datalake_results["metric"] == "recovery_correct"
].copy()

recovery_correct_df

In [ ]:
recovery_time_df = datalake_results[
    datalake_results["metric"] == "recovery_time"
].copy()

plot_scalability(
    recovery_time_df,
    "Datalake recovery time",
    "Milliseconds",
)

## 7.4 Almacenamiento del datalake

Se analizan:

- `disk_bytes`
- `file_count`
- `dir_count`

In [ ]:
disk_df = datalake_results[
    datalake_results["metric"].isin([
        "disk_bytes",
        "file_count",
        "dir_count",
    ])
].copy()

disk_df

In [ ]:
disk_bytes_df = disk_df[
    disk_df["metric"] == "disk_bytes"
].copy()

disk_bytes_df["median_mb"] = (
    disk_bytes_df["median_value"] / (1024 ** 2)
)

plot_scalability(
    disk_bytes_df.rename(
        columns={
            "median_mb": "median_value"
        }
    ),
    "Datalake disk usage",
    "MB",
)

In [ ]:
for metric, ylabel in [
    ("file_count", "Files"),
    ("dir_count", "Directories"),
]:

    plot_scalability(
        disk_df[
            disk_df["metric"] == metric
        ],
        metric,
        ylabel,
    )

## 7.5 Memoria máxima del datalake

La métrica `peak_rss` procede de `/usr/bin/time -v` y está almacenada en bytes.

In [ ]:
datalake_memory = datalake_results[
    datalake_results["metric"] == "peak_rss"
].copy()

datalake_memory["median_mb"] = (
    datalake_memory["median_value"] / (1024 ** 2)
)

plot_scalability(
    datalake_memory.rename(
        columns={
            "median_mb": "median_value"
        }
    ),
    "Datalake peak RSS",
    "MB",
)

# 8. Índices

Se comparan las estructuras:

- `json`
- `mongo`
- `folders`

Métricas:

- `build_time`
- `query_mean`
- `query_p95`
- `update_50_time`
- `disk_bytes`
- `peak_rss`

In [ ]:
index_results = all_results[
    all_results["experiment"] == "index"
].copy()

index_results

In [ ]:
for metric in [
    "build_time",
    "query_mean",
    "query_p95",
    "update_50_time",
]:

    plot_scalability(
        index_results[
            index_results["metric"] == metric
        ],
        metric,
        "Milliseconds",
    )

In [ ]:
index_disk = index_results[
    index_results["metric"] == "disk_bytes"
].copy()

index_disk["median_mb"] = (
    index_disk["median_value"] / (1024 ** 2)
)

plot_scalability(
    index_disk.rename(
        columns={
            "median_mb": "median_value"
        }
    ),
    "Index disk usage",
    "MB",
)

In [ ]:
index_memory = index_results[
    index_results["metric"] == "peak_rss"
].copy()

index_memory["median_mb"] = (
    index_memory["median_value"] / (1024 ** 2)
)

plot_scalability(
    index_memory.rename(
        columns={
            "median_mb": "median_value"
        }
    ),
    "Index peak RSS",
    "MB",
)

# 9. Descarga

La descarga HTTP se analiza separadamente de los experimentos sobre el corpus local.

Métrica:

- `http_throughput`

También se registra:

- `peak_rss`

In [ ]:
download_results = all_results[
    all_results["experiment"] == "download"
].copy()

download_results

In [ ]:
download_throughput = download_results[
    download_results["metric"] == "http_throughput"
].copy()

plot_scalability(
    download_throughput,
    "Gutenberg download throughput",
    "Books per second",
)

In [ ]:
download_memory = download_results[
    download_results["metric"] == "peak_rss"
].copy()

download_memory["median_mb"] = (
    download_memory["median_value"] / (1024 ** 2)
)

plot_scalability(
    download_memory.rename(
        columns={
            "median_mb": "median_value"
        }
    ),
    "Download peak RSS",
    "MB",
)

# 10. Escalabilidad

Las tablas siguientes permiten observar la evolución de cada métrica al aumentar el corpus:

`100 → 250 → 500 → 1000 libros`.

No se agregan conclusiones automáticas sobre qué configuración es mejor.

In [ ]:
def scalability_table(
    dataframe: pd.DataFrame,
    metric: str,
) -> pd.DataFrame:

    filtered = dataframe[
        dataframe["metric"] == metric
    ]

    if filtered.empty:
        return pd.DataFrame()

    return (
        filtered
        .pivot_table(
            index="n_books",
            columns=[
                "language",
                "structure",
            ],
            values="median_value",
        )
        .sort_index()
    )

In [ ]:
scalability_table(
    datalake_results,
    "write_throughput",
)

In [ ]:
scalability_table(
    index_results,
    "query_mean",
)

# 11. Tablas para el informe

Las siguientes tablas presentan los resultados medianos de los runs oficiales.

Los valores originales de los CSV permanecen sin modificar.

In [ ]:
def report_table(
    dataframe: pd.DataFrame,
    metric: str,
) -> pd.DataFrame:

    columns = [
        "language",
        "structure",
        "n_books",
        "median_value",
        "unit",
    ]

    return (
        dataframe[
            dataframe["metric"] == metric
        ][columns]
        .sort_values(
            [
                "language",
                "structure",
                "n_books",
            ]
        )
        .reset_index(drop=True)
    )

In [ ]:
report_table(
    datalake_results,
    "write_throughput",
)

In [ ]:
report_table(
    index_results,
    "query_mean",
)

In [ ]:
report_table(
    index_results,
    "query_p95",
)

In [ ]:
report_table(
    index_results,
    "build_time",
)

In [ ]:
report_table(
    datalake_results,
    "disk_bytes",
)

# 12. Resumen de cobertura

Esta comprobación final confirma que el notebook ha cargado los tres lenguajes y los tres experimentos oficiales.

In [ ]:
coverage = (
    all_results
    .groupby(
        ["language", "experiment"],
        as_index=False,
    )
    .agg(
        structures=("structure", "nunique"),
        sizes=("n_books", "nunique"),
        metrics=("metric", "nunique"),
    )
)

coverage

# 13. Conclusiones

Las conclusiones se redactarán después de ejecutar los benchmarks oficiales y revisar los resultados obtenidos.

Este notebook no determina automáticamente una configuración, lenguaje o estructura como "mejor".

Las conclusiones del informe deberán describir las diferencias observadas en los datos y relacionarlas con las métricas definidas en el SPEC.